# Задание 13. Калибровка оценки риска

Калибровка выполняется на валидационной выборке SMS и на собственном русско-казахском наборе. Для валидации SMS-модель обучается заново только на train, чтобы вероятности не были получены на данных, которые модель уже видела. Тестовая выборка не используется.

In [1]:
# Настройка путей. Работает локально и в Google Colab.
import sys
import zipfile
from pathlib import Path

if "google.colab" in sys.modules and not Path("ml").exists():
    # В Colab загрузите архив проекта sms_guard.zip (см. README).
    from google.colab import files
    uploaded = files.upload()
    zipfile.ZipFile(next(iter(uploaded))).extractall(".")

ROOT = Path.cwd() if (Path.cwd() / "ml").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
FIGURES = ROOT / "docs" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
print("Корень проекта:", ROOT.name)

Корень проекта: AIproject


In [2]:
import copy
import itertools
import json

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

from ml.data.split_sms import load_datasets
from ml.features import build_sms_pipeline
from ml.language import TRAINED_LANGUAGES, detect_language
from ml.predictor import load_sms_meta, load_url_meta, predict_url
from ml.risk import compute_risk, load_config
from ml.rules import find_rule_groups
from ml.url_features import extract_url_features, extract_urls, model_features, triggered_features, validate_url

data = load_datasets()
c_value = float(load_sms_meta()["params"]["clf__C"])
sms_model = build_sms_pipeline(LogisticRegression(C=c_value, class_weight="balanced", max_iter=2000, random_state=42))
sms_model.fit(data["train"]["text_raw"], data["train"]["label"])
T2 = load_url_meta()["t2"]

def prepare(texts, labels, source):
    rows = []
    probabilities = sms_model.predict_proba(list(texts))[:, 1]
    for text, label, p_sms in zip(texts, labels, probabilities):
        language = detect_language(text)
        urls = [u for u in extract_urls(text) if validate_url(u)[1] is None]
        p_urls = [predict_url(model_features(u)) for u in urls]
        flags = [code for u in urls for code in triggered_features(extract_url_features(u))]
        rows.append({
            "source": source, "label": label, "language": language,
            "p_sms": float(p_sms) if language in TRAINED_LANGUAGES else None,
            "p_url_max": max(p_urls) if p_urls else None,
            "groups": find_rule_groups(text), "flags": flags,
        })
    return rows

custom = pd.read_csv(ROOT / "data" / "raw" / "custom_ru_kk.csv", encoding="utf-8")
items = pd.DataFrame(
    prepare(data["val"]["text_raw"], data["val"]["label"], "val (en)")
    + prepare(custom["text"], (custom["label"] == "fraud").astype(int), "свой набор (ru/kk)")
)
items.groupby("source")["label"].agg(["count", "sum"]).rename(columns={"count": "Записей", "sum": "Мошеннических"})

,Записей,Мошеннических
source,,
val (en),770,92
свой набор (ru/kk),150,75


In [3]:
def none_if_nan(value):
    return None if value is None or pd.isna(value) else float(value)


def evaluate(config):
    levels = [
        compute_risk(none_if_nan(r.p_sms), none_if_nan(r.p_url_max), r.groups, url_flags=r.flags, url_t2=T2, config=config).level
        for r in items.itertuples()
    ]
    frame = items.assign(level=levels)
    fraud, safe = frame[frame.label == 1], frame[frame.label == 0]
    result = {
        "Recall (средний и выше)": fraud["level"].isin(["MEDIUM", "HIGH"]).mean(),
        "Доля безопасных с высоким": (safe["level"] == "HIGH").mean(),
        "Доля безопасных со средним и выше": safe["level"].isin(["MEDIUM", "HIGH"]).mean(),
    }
    for source, part in frame.groupby("source"):
        result[f"Recall, {source}"] = part.loc[part.label == 1, "level"].isin(["MEDIUM", "HIGH"]).mean()
        result[f"Безоп. высокий, {source}"] = (part.loc[part.label == 0, "level"] == "HIGH").mean()
    return result

base = copy.deepcopy(load_config())
print("Начальные значения из ТЗ:")
pd.Series(evaluate(base)).round(3)

Начальные значения из ТЗ:


Recall (средний и выше)               0.952
Доля безопасных с высоким             0.001
Доля безопасных со средним и выше     0.004
Recall, val (en)                      0.935
Безоп. высокий, val (en)              0.001
Recall, свой набор (ru/kk)            0.973
Безоп. высокий, свой набор (ru/kk)    0.000
dtype: float64

## Перебор вариантов

Перебираются веса `w1` (текст), `w2` (ссылка), `w3` (социальная инженерия), насыщение `S_se` и границы уровней. Подходящим считается вариант с Recall ≥ 0,95 и долей безопасных с высоким уровнем ≤ 3 %. Среди подходящих выбирается вариант с наибольшим Recall (приоритет из раздела 3.4), затем с наименьшей долей безопасных сообщений со средним уровнем и выше; при равенстве — с границей высокого уровня 70, как в ТЗ.

In [4]:
rows = []
for w1, w2, w3, saturation, medium, high in itertools.product(
    [0.4, 0.5, 0.6], [0.2, 0.3, 0.4], [0.2, 0.3, 0.4], [2, 3], [25, 30, 35], [60, 70]
):
    config = copy.deepcopy(base)
    config["weights"] = {"sms": w1, "url": w2, "social_engineering": w3}
    config["social_engineering_saturation"] = saturation
    config["levels"] = {"medium_from": medium, "high_from": high}
    rows.append({"w1": w1, "w2": w2, "w3": w3, "S_se": saturation, "Средний от": medium, "Высокий от": high, **evaluate(config)})
calibration = pd.DataFrame(rows)
ok = calibration[(calibration["Recall (средний и выше)"] >= 0.95) & (calibration["Доля безопасных с высоким"] <= 0.03)]
print(f"Вариантов всего: {len(calibration)}, подходящих: {len(ok)}")
ranked = (ok if len(ok) else calibration).sort_values(
    ["Recall (средний и выше)", "Доля безопасных со средним и выше", "Высокий от", "w1"],
    ascending=[False, True, False, False],
    kind="stable",
)
ranked.head(10).round(3)

Вариантов всего: 324, подходящих: 278


,w1,w2,w3,S_se,Средний от,Высокий от,Recall (средний и выше),Доля безопасных с высоким,Доля безопасных со средним и выше,"Recall, val (en)","Безоп. высокий, val (en)","Recall, свой набор (ru/kk)","Безоп. высокий, свой набор (ru/kk)"
229,0.6,0.2,0.3,2,25,70,0.976,0.001,0.004,0.957,0.001,1.0,0.0
235,0.6,0.2,0.3,3,25,70,0.976,0.001,0.004,0.957,0.001,1.0,0.0
265,0.6,0.3,0.3,2,25,70,0.976,0.001,0.004,0.957,0.001,1.0,0.0
271,0.6,0.3,0.3,3,25,70,0.976,0.001,0.004,0.957,0.001,1.0,0.0
301,0.6,0.4,0.3,2,25,70,0.976,0.001,0.004,0.957,0.001,1.0,0.0
307,0.6,0.4,0.3,3,25,70,0.976,0.001,0.004,0.957,0.001,1.0,0.0
109,0.5,0.2,0.2,2,25,70,0.976,0.001,0.004,0.957,0.001,1.0,0.0
115,0.5,0.2,0.2,3,25,70,0.976,0.001,0.004,0.957,0.001,1.0,0.0
145,0.5,0.3,0.2,2,25,70,0.976,0.001,0.004,0.957,0.001,1.0,0.0
151,0.5,0.3,0.2,3,25,70,0.976,0.001,0.004,0.957,0.001,1.0,0.0


In [5]:
best = ranked.iloc[0]
chosen = {
    "weights": {"sms": float(best.w1), "url": float(best.w2), "social_engineering": float(best.w3)},
    "social_engineering_saturation": int(best.S_se),
    "levels": {"medium_from": int(best["Средний от"]), "high_from": int(best["Высокий от"])},
}
(ROOT / "docs" / "13_risk_calibration.json").write_text(json.dumps(chosen, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(chosen, ensure_ascii=False, indent=2))
final_config = copy.deepcopy(base)
final_config.update(chosen)
pd.DataFrame({"Начальные (ТЗ)": evaluate(base), "Выбранные": evaluate(final_config)}).round(3)

{
  "weights": {
    "sms": 0.6,
    "url": 0.2,
    "social_engineering": 0.3
  },
  "social_engineering_saturation": 2,
  "levels": {
    "medium_from": 25,
    "high_from": 70
  }
}


,Начальные (ТЗ),Выбранные
Recall (средний и выше),0.952,0.976
Доля безопасных с высоким,0.001,0.001
Доля безопасных со средним и выше,0.004,0.004
"Recall, val (en)",0.935,0.957
"Безоп. высокий, val (en)",0.001,0.001
"Recall, свой набор (ru/kk)",0.973,1.000
"Безоп. высокий, свой набор (ru/kk)",0.000,0.000
